# Izhikevich PINN Training Notebook

**Self-contained** notebook for training the Physics-Informed Neural Network on Google Colab.

## Instructions
1. **Runtime → Change runtime type → GPU (T4)**
2. Upload `ground_truth.csv` to a folder on your Google Drive
3. Set `DRIVE_FOLDER` in Cell 1 to point to that folder
4. Run all cells — the model and plots are saved back to your Drive

---

## Cell 1: Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# ============================================================
# SET YOUR FOLDER PATH HERE
# ============================================================
# This should be the folder on your Drive containing ground_truth.csv
# Example: 'My Drive/Izhikevich-Dynamic-Neuron-Model/data'
DRIVE_FOLDER = 'My Drive/Izhikevich-Dynamic-Neuron-Model/data'

# Full paths
drive_path = f'/content/drive/{DRIVE_FOLDER}'
csv_filename = os.path.join(drive_path, 'ground_truth.csv')

# Verify
assert os.path.exists(csv_filename), (
    f'File not found: {csv_filename}\n'
    f'Please check your DRIVE_FOLDER path above.'
)
print(f'CSV found: {csv_filename}')
print(f'File size: {os.path.getsize(csv_filename) / 1e6:.1f} MB')

## Cell 2: Verify GPU & Environment

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU")

# Quick CSV check
df = pd.read_csv(csv_filename)
print(f"\nCSV shape: {df.shape}")
print(f"Columns:   {list(df.columns)}")
print(f"Time:      {df['Time (ms)'].min()} to {df['Time (ms)'].max()} ms")

## Cell 3: Configuration (from config.py)

In [ ]:
# ============================================================
# CONFIG — Izhikevich 2007 Biophysical Model Parameters
# ============================================================

# --- 2007 Biophysical Parameters ---
C_m = 100.0     # Membrane capacitance (pF)
k = 0.7         # Scaling constant (nS/mV)
v_r = -60.0     # Resting membrane potential (mV)
v_t = -40.0     # Instantaneous threshold potential (mV)

# --- Dimensionless / Reset Parameters ---
a = 0.03        # Recovery time-scale (1/ms)
b = -2.0        # Recovery sensitivity (nS)
c = -50.0       # After-spike reset of v (mV)
d = 100.0       # After-spike reset increment of w (pA)

# --- Spike Detection ---
v_peak = 35.0   # Spike cutoff / peak voltage (mV)

# --- Initial Conditions ---
V_0 = -60.0
W_0 = 0.0
INITIAL_STATE = np.array([V_0, W_0])

# --- Simulation Settings ---
T_START = 0.0
T_END = 1000.0
DT_EVAL = 0.01

# --- Step-current protocol ---
I_EXT_DEFAULT = 70.0
T_STIM_ONSET = 100.0

def I_ext_fn(t):
    return np.where(np.asarray(t) >= T_STIM_ONSET, I_EXT_DEFAULT, 0.0)

print("Config loaded.")
print(f"  Parameters: C_m={C_m}, k={k}, v_r={v_r}, v_t={v_t}")
print(f"  Reset: a={a}, b={b}, c={c}, d={d}, v_peak={v_peak}")
print(f"  Stimulus: {I_EXT_DEFAULT} pA onset at {T_STIM_ONSET} ms")
print(f"  Time: {T_START} to {T_END} ms, dt={DT_EVAL}")

## Cell 4: Architecture (IzhikevichPINN)

In [ ]:
import torch.nn as nn

# Denormalization constants
V_SHIFT = v_r                   # -60.0 mV
V_SCALE = v_peak - v_r          #  95.0 mV
W_SHIFT = 0.0                   #   0.0 pA
W_SCALE = abs(d)                # 100.0 pA


class IzhikevichPINN(nn.Module):
    """PINN for the Izhikevich model.
    Input:  t (N,1) in ms. Output: [v, w] (N,2) in physical units.
    """

    def __init__(self, hidden_size=128, num_layers=6):
        super().__init__()
        self.register_buffer('v_scale', torch.tensor(V_SCALE, dtype=torch.float32))
        self.register_buffer('v_shift', torch.tensor(V_SHIFT, dtype=torch.float32))
        self.register_buffer('w_scale', torch.tensor(W_SCALE, dtype=torch.float32))
        self.register_buffer('w_shift', torch.tensor(W_SHIFT, dtype=torch.float32))
        self.register_buffer('t_max', torch.tensor(T_END, dtype=torch.float32))

        layers = []
        layers.append(nn.Linear(1, hidden_size))
        layers.append(nn.Tanh())
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_size, 2))
        self.network = nn.Sequential(*layers)

        # Xavier initialization
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.0)
                nn.init.zeros_(m.bias)

    def forward(self, t):
        t_norm = (t - self.t_max / 2) / (self.t_max / 2)
        raw = self.network(t_norm)
        v = raw[:, 0:1] * self.v_scale + self.v_shift
        w = raw[:, 1:2] * self.w_scale + self.w_shift
        return torch.cat([v, w], dim=1)


def predict_trajectory(model, t_start=T_START, t_end=T_END, dt=DT_EVAL):
    """Generate (N, 3) array [Time, v, w] from trained model."""
    time_np = np.arange(t_start, t_end + dt * 0.5, dt)
    t_tensor = torch.tensor(time_np, dtype=torch.float32).view(-1, 1)
    model.eval()
    with torch.no_grad():
        device = next(model.parameters()).device
        t_tensor = t_tensor.to(device)
        outputs = model(t_tensor).cpu().numpy()
    return np.column_stack([time_np, outputs])


# Quick test
test_model = IzhikevichPINN()
print(f"Architecture OK. Parameters: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model

## Cell 5: Physics Loss

In [ ]:
# ============================================================
# PHYSICS-INFORMED LOSS FUNCTIONS
# ============================================================

def I_ext_torch(t):
    """Step-current stimulus for PyTorch tensors."""
    return I_EXT_DEFAULT * (t >= T_STIM_ONSET).float()

# Characteristic ODE rates for residual normalization
DV_RATE = I_EXT_DEFAULT / C_m       # 0.7  mV/ms
DW_RATE = a * abs(d)                # 3.0  pA/ms


def compute_physics_loss(model, t, peak_margin=10.0):
    """ODE-residual loss with spike-masking."""
    outputs = model(t)
    v_pred = outputs[:, 0:1]
    w_pred = outputs[:, 1:2]

    dv_dt = torch.autograd.grad(
        outputs=v_pred, inputs=t,
        grad_outputs=torch.ones_like(v_pred),
        create_graph=True, retain_graph=True,
    )[0]

    dw_dt = torch.autograd.grad(
        outputs=w_pred, inputs=t,
        grad_outputs=torch.ones_like(w_pred),
        create_graph=True, retain_graph=True,
    )[0]

    I_ext = I_ext_torch(t.detach())

    rhs_v = (k * (v_pred - v_r) * (v_pred - v_t) - w_pred + I_ext) / C_m
    rhs_w = a * (b * (v_pred - v_r) - w_pred)

    R_v = dv_dt - rhs_v
    R_w = dw_dt - rhs_w

    valid = ~(v_pred > (v_peak - peak_margin)).squeeze()
    R_v_valid = R_v[valid]
    R_w_valid = R_w[valid]

    if R_v_valid.numel() == 0:
        return torch.tensor(0.0, device=t.device, requires_grad=True)

    return (
        torch.mean((R_v_valid / DV_RATE) ** 2)
        + torch.mean((R_w_valid / DW_RATE) ** 2)
    )


def compute_ic_loss(model):
    """Initial-condition loss at t=0."""
    IC_V_SCALE = v_peak - v_r
    IC_W_SCALE = abs(d)
    ic_V_0 = float(INITIAL_STATE[0])
    ic_W_0 = float(INITIAL_STATE[1])

    device = next(model.parameters()).device
    t_zero = torch.zeros(1, 1, device=device)
    out = model(t_zero)
    return (
        ((out[:, 0:1] - ic_V_0) / IC_V_SCALE) ** 2
        + ((out[:, 1:2] - ic_W_0) / IC_W_SCALE) ** 2
    ).mean()


def get_margin(epoch, max_epochs, start=20.0, end=5.0):
    """Curriculum margin: wide early, tight late."""
    progress = min(epoch / max(max_epochs, 1), 1.0)
    return start + (end - start) * progress


print("Physics loss functions loaded.")
print(f"  DV_RATE = {DV_RATE:.2f} mV/ms")
print(f"  DW_RATE = {DW_RATE:.2f} pA/ms")

## Cell 6: Training Configuration

**Edit these values before running if you want to experiment:**

In [ ]:
# ============================================================
# TRAINING HYPERPARAMETERS — edit these!
# ============================================================

DATA_SUBSAMPLE_FACTOR = 100   # Every Nth point for data loss (100 = ~1K pts)
PHYSICS_BATCH_SIZE = 16384    # Collocation pts per physics batch (VRAM control)
ADAM_EPOCHS = 8000             # Phase 1 epochs
LBFGS_EPOCHS = 100            # Phase 2 epochs
LR_ADAM = 1e-3                # Adam learning rate
LR_LBFGS = 0.1                # L-BFGS learning rate

# Loss weights
LAMBDA_PHYS = 1.0
LAMBDA_IC = 200.0
LAMBDA_DATA = 50.0

MODEL_SAVE_NAME = f"pinn_model_dsf{DATA_SUBSAMPLE_FACTOR}.pt"

print(f"Data subsample factor: {DATA_SUBSAMPLE_FACTOR}")
print(f"Physics batch size:    {PHYSICS_BATCH_SIZE}")
print(f"Epochs: {ADAM_EPOCHS} Adam + {LBFGS_EPOCHS} L-BFGS")
print(f"Model will be saved as: {MODEL_SAVE_NAME}")

## Cell 7: Run Training

In [ ]:
import torch.optim as optim
import time as _time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ---- 1. Create Run Folder & Save Parameters ----
run_folder_name = f"run_dsf{DATA_SUBSAMPLE_FACTOR}"
run_path = os.path.join(drive_path, run_folder_name)
os.makedirs(run_path, exist_ok=True)

params = {
    "DATA_SUBSAMPLE_FACTOR": DATA_SUBSAMPLE_FACTOR,
    "PHYSICS_BATCH_SIZE": PHYSICS_BATCH_SIZE,
    "ADAM_EPOCHS": ADAM_EPOCHS,
    "LBFGS_EPOCHS": LBFGS_EPOCHS,
    "LR_ADAM": LR_ADAM,
    "LR_LBFGS": LR_LBFGS,
    "LAMBDA_PHYS": LAMBDA_PHYS,
    "LAMBDA_IC": LAMBDA_IC,
    "LAMBDA_DATA": LAMBDA_DATA,
    "C_m": C_m, "k": k, "v_r": v_r, "v_t": v_t,
    "a": a, "b": b, "c": c, "d": d, "v_peak": v_peak,
}
with open(os.path.join(run_path, "parameters.txt"), "w") as f:
    for param_k, param_v in params.items():
        f.write(f"{param_k}: {param_v}\n")

# ---- 2. Load Data ----
df = pd.read_csv(csv_filename)
all_t    = torch.tensor(df['Time (ms)'].values, dtype=torch.float32).unsqueeze(1).to(device)
all_v_gt = torch.tensor(df['v (mV)'].values,    dtype=torch.float32).unsqueeze(1).to(device)
all_w_gt = torch.tensor(df['w (pA)'].values,    dtype=torch.float32).unsqueeze(1).to(device)
n_total = len(all_t)

# ---- 3. Sparse Data vs Dense Collocation ----
colloc_t = all_t
sparse_idx = torch.arange(0, n_total, DATA_SUBSAMPLE_FACTOR)
data_t = all_t[sparse_idx]
data_targets = torch.cat([all_v_gt[sparse_idx], all_w_gt[sparse_idx]], dim=1)
n_data = len(sparse_idx)
n_colloc = len(colloc_t)

print(f"Total samples:     {n_total:,}")
print(f"Physics colloc:    {n_total:,} (batched {PHYSICS_BATCH_SIZE})")
print(f"Data supervision:  {n_data:,} (every {DATA_SUBSAMPLE_FACTOR}th)")
print(f"Data/Physics ratio: 1:{DATA_SUBSAMPLE_FACTOR}")

# ---- 4. Model ----
model = IzhikevichPINN().to(device)
mse_criterion = nn.MSELoss()
print(f"Model parameters:  {sum(p.numel() for p in model.parameters()):,}")

total_epochs = ADAM_EPOCHS + LBFGS_EPOCHS

# ==================================================================
# Phase 1 — Adam (batched physics, full-batch data)
# ==================================================================
optimizer_adam = optim.Adam(model.parameters(), lr=LR_ADAM)
model.train()

print(f"\n{'='*60}")
print(f"Phase 1: Adam ({ADAM_EPOCHS} epochs)")
print(f"  Physics: {n_total:,} collocation pts (batched {PHYSICS_BATCH_SIZE})")
print(f"  Data:    {n_data:,} sparse pts (full-batch)")
print(f"{'='*60}")

t_start_wall = _time.time()
loss_history = []

for epoch in range(1, ADAM_EPOCHS + 1):
    epoch_loss_total = 0.0
    epoch_loss_data = 0.0
    epoch_loss_phys = 0.0
    epoch_loss_ic = 0.0
    n_batches = 0

    # Shuffle collocation points each epoch
    perm = torch.randperm(n_colloc, device=device)

    for i in range(0, n_colloc, PHYSICS_BATCH_SIZE):
        idx = perm[i:i + PHYSICS_BATCH_SIZE]
        t_batch = colloc_t[idx].clone().requires_grad_(True)

        # -- Physics loss (batched collocation) --
        margin = get_margin(epoch, total_epochs)
        loss_phys = compute_physics_loss(
            model, t_batch, peak_margin=margin
        )

        # -- Data loss (full sparse set) --
        pred_data = model(data_t)                     # (M, 2)
        loss_data = mse_criterion(pred_data, data_targets)

        # -- IC loss --
        loss_ic = compute_ic_loss(model)

        # -- Total --
        total_loss = (
            LAMBDA_DATA * loss_data
            + LAMBDA_PHYS * loss_phys
            + LAMBDA_IC * loss_ic
        )

        optimizer_adam.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_adam.step()

        epoch_loss_total += total_loss.item()
        epoch_loss_data += loss_data.item()
        epoch_loss_phys += loss_phys.item()
        epoch_loss_ic += loss_ic.item()
        n_batches += 1

    nb = max(n_batches, 1)
    loss_history.append(epoch_loss_total / nb)

    if epoch % 500 == 0 or epoch == 1:
        elapsed = _time.time() - t_start_wall
        print(
            f"  Epoch [{epoch:4d}/{ADAM_EPOCHS}]  "
            f"Total: {epoch_loss_total/nb:.4e}  "
            f"Data: {epoch_loss_data/nb:.4e}  "
            f"Phys: {epoch_loss_phys/nb:.4e}  "
            f"IC: {epoch_loss_ic/nb:.4e}  "
            f"Margin: {margin:.1f}  "
            f"({elapsed:.0f}s)"
        )

adam_time = _time.time() - t_start_wall
print(f"\nAdam phase completed in {adam_time:.1f}s")

# ==================================================================
# Phase 2 — L-BFGS (fixed physics subsample + full-batch data)
# ==================================================================
print(f"\n{'='*60}")
print(f"Phase 2: L-BFGS refinement ({LBFGS_EPOCHS} epochs)")
print(f"{'='*60}")

# Fixed RANDOM subsample for L-BFGS (full 100K would OOM with autograd)
n_lbfgs = min(PHYSICS_BATCH_SIZE, n_colloc)
lbfgs_idx = torch.randperm(n_colloc, device=device)[:n_lbfgs]
lbfgs_t = colloc_t[lbfgs_idx].clone().requires_grad_(True)

optimizer_lbfgs = optim.LBFGS(
    model.parameters(), lr=LR_LBFGS,
    max_iter=20, history_size=50,
    line_search_fn="strong_wolfe",
)

t_start_lbfgs = _time.time()

for lbfgs_epoch in range(1, LBFGS_EPOCHS + 1):

    def closure():
        optimizer_lbfgs.zero_grad()

        margin = get_margin(ADAM_EPOCHS + lbfgs_epoch, total_epochs)
        loss_phys = compute_physics_loss(
            model, lbfgs_t, peak_margin=margin,
        )

        pred_data = model(data_t)
        loss_data = mse_criterion(pred_data, data_targets)

        loss_ic = compute_ic_loss(model)

        total = (
            LAMBDA_DATA * loss_data
            + LAMBDA_PHYS * loss_phys
            + LAMBDA_IC * loss_ic
        )
        total.backward()
        return total

    loss_val = optimizer_lbfgs.step(closure)

    if lbfgs_epoch % 20 == 0 or lbfgs_epoch == 1:
        print(f"  L-BFGS [{lbfgs_epoch:4d}/{LBFGS_EPOCHS}]  Loss: {loss_val.item():.6f}")

lbfgs_time = _time.time() - t_start_lbfgs
total_time = _time.time() - t_start_wall
print(f"\nL-BFGS phase completed in {lbfgs_time:.1f}s")
print(f"Total training time: {total_time:.1f}s ({total_time/60:.1f} min)")

# ==================================================================
# Save Model & Prediction CSV
# ==================================================================
model_save_path = os.path.join(run_path, MODEL_SAVE_NAME)
torch.save(model.state_dict(), model_save_path)
print(f"\nModel saved to: {model_save_path}")

# Generate and save prediction CSV
model.eval()
results = predict_trajectory(model)
print(f"Output trajectory shape: {results.shape}  (expected (N, 3))")

pred_df = pd.DataFrame({
    'Time (ms)': results[:, 0],
    'v (mV)': results[:, 1],
    'w (pA)': results[:, 2],
})
pred_csv_path = os.path.join(run_path, f'prediction_dsf{DATA_SUBSAMPLE_FACTOR}.csv')
pred_df.to_csv(pred_csv_path, index=False)
print(f"Prediction CSV saved to: {pred_csv_path}")
print(f"\nAll results saved to: {run_path}")

## Cell 8: Loss Curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.semilogy(loss_history, lw=1)
plt.xlabel('Epoch')
plt.ylabel('Total Loss (log scale)')
plt.title('Adam Training Loss Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
loss_curve_path = os.path.join(run_path, 'loss_curve.png')
plt.savefig(loss_curve_path, dpi=150, bbox_inches='tight')
print(f"Loss curve saved to: {loss_curve_path}")
plt.show()

## Cell 9: Verification — PINN vs Ground Truth

In [ ]:
# Ground truth
t_gt = df['Time (ms)'].values
v_gt = df['v (mV)'].values
w_gt = df['w (pA)'].values

# PINN prediction (already generated in Cell 7)
t_pred = results[:, 0]
v_pred = results[:, 1]
w_pred = results[:, 2]

# ---- Plot v(t) and w(t) ----
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(t_gt, v_gt, 'k--', lw=2, label='Ground Truth', alpha=0.7)
axes[0].plot(t_pred, v_pred, 'r-', lw=1.5, label='PINN Prediction', alpha=0.8)
axes[0].set_ylabel('v (mV)')
axes[0].set_title(f'Izhikevich PINN Verification (dsf={DATA_SUBSAMPLE_FACTOR})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_gt, w_gt, 'k--', lw=2, label='Ground Truth', alpha=0.7)
axes[1].plot(t_pred, w_pred, 'b-', lw=1.5, label='PINN Prediction', alpha=0.8)
axes[1].set_ylabel('w (pA)')
axes[1].set_xlabel('Time (ms)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot to run folder
plot_save_path = os.path.join(run_path, f'verify_pinn_dsf{DATA_SUBSAMPLE_FACTOR}.png')
plt.savefig(plot_save_path, dpi=150, bbox_inches='tight')
print(f"Plot saved to: {plot_save_path}")
plt.show()

# ---- RMSE ----
n = min(len(v_pred), len(v_gt))
rmse_v = np.sqrt(np.mean((v_pred[:n] - v_gt[:n])**2))
rmse_w = np.sqrt(np.mean((w_pred[:n] - w_gt[:n])**2))
print(f"\nRMSE v: {rmse_v:.4f} mV")
print(f"RMSE w: {rmse_w:.4f} pA")

print(f"\n--- All results saved to: {run_path} ---")